In [5]:
import sys
import json
from pathlib import Path
from collections import Counter

import pandas as pd

import math
from scipy import stats


In [8]:
TEST_TYPE_ALIASES = {
    "t": "t",
    "f": "F",
    "chi": "chi", "chi2": "chi", "chisq": "chi", "chi-square": "chi", "chi_square": "chi",
    "z": "z",
    "r": "r",
    "q": "Q",
}


# Анализ согласованности стат. тестов

LLM на этом этапе уже не нужен — считаем
p-value через scipy и сравниваем с тем, что заявили авторы.

Что хотим увидеть:
- `compute_p` корректно считает для F / t / Chi2 / r;
- `check_consistency` ловит расхождения reported_p vs computed_p;
- `check_interpretation` ловит расхождения direction vs computed_p;
- `summary` собирает сводку по всей статье.

In [2]:
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))


In [4]:
GOLD_TESTS = ROOT / 'stress_burnout_workplace_psychosocial_factors_mental_health_covid19_hungary.json'
gold_tests = json.loads(GOLD_TESTS.read_text(encoding='utf-8'))

print(f'Тестов в разметке: {len(gold_tests)}')
print('По типу:', dict(Counter(t['test_type'] for t in gold_tests)))

Тестов в разметке: 41
По типу: {'F': 21, 'Chi2': 4, 'r': 8, 't': 8}


In [6]:

def normalize_test_type(test_type):
    if test_type is None:
        return None
    return TEST_TYPE_ALIASES.get(str(test_type).strip().lower())


# Пересчёт p-value из статистики теста (scipy.stats)
def compute_p(test_type, statistic_value, df1, df2, two_tailed=True):
    if statistic_value is None:
        return None
    tt = normalize_test_type(test_type)

    try:
        if tt == "t":
            if df1 is None:
                return None
            p = stats.t.sf(abs(statistic_value), df1) * (2 if two_tailed else 1)

        elif tt == "F":
            if df1 is None or df2 is None:
                return None
            p = stats.f.sf(statistic_value, df1, df2)

        elif tt == "chi":
            if df1 is None:
                return None
            p = stats.chi2.sf(statistic_value, df1)

        elif tt == "z":
            p = stats.norm.sf(abs(statistic_value)) * (2 if two_tailed else 1)

        elif tt == "r":
            if df1 is None:
                return None
            denom = 1 - statistic_value ** 2
            if denom <= 0:
                return None
            t_val = statistic_value * math.sqrt(df1) / math.sqrt(denom)
            p = stats.t.sf(abs(t_val), df1) * (2 if two_tailed else 1)

        elif tt == "Q":
            if df1 is None:
                return None
            p = stats.chi2.sf(statistic_value, df1)

        else:
            return None

        return round(p, 10)

    except (ValueError, ZeroDivisionError):
        return None



In [9]:
t = gold_tests[0]
p = compute_p(t['test_type'], t['statistic_value'], t['df1'], t['df2'])

print(f"raw: {t['raw_text']}")
print(f"computed_p = {p:.6f}  vs  reported_p {t['p_equality']} {t['reported_p']}")

# для F(1, 266)=11.816 ожидаем p ≈ 0.0007 — тот же порядок, что в статье
assert p is not None and p < 0.01, 'compute_p сломался для F-теста'

raw: Work pace: F=11.816, Sig.=0.001 (COVID care 59.08 vs Not 49.78)
computed_p = 0.000681  vs  reported_p = 0.001


In [10]:
for test_type in ['F', 't', 'r', 'Chi2']:
    samples = [x for x in gold_tests if x['test_type'] == test_type]
    sample = next((x for x in samples if x.get('df1') is not None), samples[0])
    p = compute_p(sample['test_type'], sample['statistic_value'], sample['df1'], sample['df2'])
    print(f"{test_type:5s} stat={sample['statistic_value']:>8} df1={sample['df1']} df2={sample['df2']} -> computed_p={p}")

F     stat=  11.816 df1=1 df2=266 -> computed_p=0.0006813479
t     stat=   8.079 df1=263 df2=None -> computed_p=0.0
r     stat=    0.38 df1=266 df2=None -> computed_p=1e-10
Chi2  stat=  27.251 df1=None df2=None -> computed_p=None


In [12]:
# проверка консистентности
# tolerance задаёт относительную погрешность для меток consistent / marginal
def check_consistency(reported_p, computed_p, p_equality, tolerance=0.05):
    if reported_p is None or computed_p is None:
        return "not_checkable"

    if p_equality == "<":
        if computed_p < reported_p:
            return "consistent"
        elif abs(computed_p - reported_p) / max(reported_p, 1e-10) < tolerance:
            return "marginal"
        else:
            return "inconsistent"

    elif p_equality == ">":
        if computed_p > reported_p:
            return "consistent"
        elif abs(computed_p - reported_p) / max(reported_p, 1e-10) < tolerance:
            return "marginal"
        else:
            return "inconsistent"

    else:  # "=" или None
        ratio = abs(computed_p - reported_p) / max(reported_p, 1e-10)
        if ratio < tolerance:
            return "consistent"
        elif ratio < tolerance * 3:
            return "marginal"
        else:
            return "inconsistent"


In [19]:
# несколько кейсов на разные ветки логики
cases = [
      (0.001, 0.00099, '=',  'consistent'),
      (0.05,  0.5,    '=',  'inconsistent'),
      (0.001, 0.0001, '<',  'consistent'),
      (0.05,  0.04,   '>',  'inconsistent'),
      (None,  0.001,  '=',  'not_checkable'),
]
for reported, computed, eq, expected in cases:
    got = check_consistency(reported, computed, eq)
    print(f'reported={reported} {eq} computed={computed} -> {got}  (ожид. {expected})')
    assert got == expected

reported=0.001 = computed=0.00099 -> consistent  (ожид. consistent)
reported=0.05 = computed=0.5 -> inconsistent  (ожид. inconsistent)
reported=0.001 < computed=0.0001 -> consistent  (ожид. consistent)
reported=0.05 > computed=0.04 -> inconsistent  (ожид. inconsistent)
reported=None = computed=0.001 -> not_checkable  (ожид. not_checkable)


In [17]:
# check_interpretation — direction vs computed_p
def check_interpretation(interpretation_direction, computed_p, alpha=0.05):
    if computed_p is None:
        return "not_checkable"

    if interpretation_direction == "significant" and computed_p <= alpha:
        return "consistent"
    elif interpretation_direction == "not_significant" and computed_p > alpha:
        return "consistent"
    elif interpretation_direction == "marginal" and 0.01 < computed_p <= 0.10:
        return "consistent"
    elif interpretation_direction in ("significant", "not_significant", "marginal"):
        return "inconsistent"
    else:
        return "unclear"

In [20]:
cases = [
    ('significant',     0.001, 'consistent'),
    ('significant',     0.5,   'inconsistent'),
    ('not_significant', 0.5,   'consistent'),
    ('not_significant', 0.001, 'inconsistent'),
    ('marginal',        0.06,  'consistent'),
    ('unclear',         0.001, 'unclear'),
    ('significant',     None,  'not_checkable'),
]
for direction, computed, expected in cases:
    got = check_interpretation(direction, computed)
    print(f'direction={direction:16s} computed={computed} -> {got}  (ожид. {expected})')
    assert got == expected

direction=significant      computed=0.001 -> consistent  (ожид. consistent)
direction=significant      computed=0.5 -> inconsistent  (ожид. inconsistent)
direction=not_significant  computed=0.5 -> consistent  (ожид. consistent)
direction=not_significant  computed=0.001 -> inconsistent  (ожид. inconsistent)
direction=marginal         computed=0.06 -> consistent  (ожид. consistent)
direction=unclear          computed=0.001 -> unclear  (ожид. unclear)
direction=significant      computed=None -> not_checkable  (ожид. not_checkable)


In [21]:
# verify_all + summary — прогон по всей статье

# верификация одной записи: добавляем computed_p + p_consistency + interpretation_consistency
def verify_test(record):
    computed_p = compute_p(
        test_type=record.get("test_type"),
        statistic_value=record.get("statistic_value"),
        df1=record.get("df1"),
        df2=record.get("df2"),
        two_tailed=record.get("two_tailed", True),
    )

    p_check = check_consistency(
        reported_p=record.get("reported_p"),
        computed_p=computed_p,
        p_equality=record.get("p_equality"),
    )

    # primary_direction приходит из interpritation_extractor.aggregate;
    direction = record.get("primary_direction") or record.get("interpretation_direction") or "unclear"
    interp_check = check_interpretation(
        interpretation_direction=direction,
        computed_p=computed_p,
    )

    return {
        **record,
        "computed_p": computed_p,
        "p_consistency": p_check,
        "interpretation_consistency": interp_check,
    }

def verify_all(tests):
    return [verify_test(t) for t in tests]

def summary(verified_tests):
    total = len(verified_tests)
    checkable = [t for t in verified_tests if t["p_consistency"] != "not_checkable"]
    n_checkable = len(checkable)

    p_consistent = sum(1 for t in checkable if t["p_consistency"] == "consistent")
    p_marginal = sum(1 for t in checkable if t["p_consistency"] == "marginal")
    p_inconsistent = sum(1 for t in checkable if t["p_consistency"] == "inconsistent")

    interp_checkable = [t for t in verified_tests if t["interpretation_consistency"] != "not_checkable"]
    interp_consistent = sum(1 for t in interp_checkable if t["interpretation_consistency"] == "consistent")
    interp_inconsistent = sum(1 for t in interp_checkable if t["interpretation_consistency"] == "inconsistent")

    return {
        "total_tests": total,
        "p_checkable": n_checkable,
        "p_consistent": p_consistent,
        "p_marginal": p_marginal,
        "p_inconsistent": p_inconsistent,
        "p_consistency_rate": round(p_consistent / n_checkable, 3) if n_checkable else None,
        "interp_checkable": len(interp_checkable),
        "interp_consistent": interp_consistent,
        "interp_inconsistent": interp_inconsistent,
        "interp_consistency_rate": round(interp_consistent / len(interp_checkable), 3) if interp_checkable else None,
    }


In [28]:
#подгрузим ранее полученную разметку тестов от ллм и посмотрим на нем
merged = pd.read_csv('/content/merged.csv')
print(merged.columns.tolist())   # убедись что есть test_type, statistic_value, df1, df2,
                                    # reported_p, p_equality, primary_direction
                                    # (interpretation_direction — для сравнения с gold)

# pandas → list[dict]
records = merged.to_dict(orient='records')

# этап 5
verified = verify_all(records)
print(json.dumps(summary(verified), ensure_ascii=False, indent=2))

['article_id', 'article_title', 'journal', 'year', 'test_instance_id', 'raw_text', 'test_type', 'statistic_value', 'df1', 'df2', 'reported_p', 'p_equality', 'two_tailed', 'textual_interpretation', 'interpretation_direction', 'consistent', 'notes', 'interpretations', 'primary_direction', 'has_cross_section_conflict', 'any_hedging', 'sections']
{
  "total_tests": 41,
  "p_checkable": 41,
  "p_consistent": 32,
  "p_marginal": 1,
  "p_inconsistent": 8,
  "p_consistency_rate": 0.78,
  "interp_checkable": 41,
  "interp_consistent": 32,
  "interp_inconsistent": 0,
  "interp_consistency_rate": 0.78
}


# Результаты обернуты в test_verificator.py
# Анализ результатов этого этапа пайплайна(и всех остальных) будут в отдельном ноутбуке
